In [1]:
# imports
import numpy as np
import tifffile as tiff
from pathlib import Path
from scipy.signal import find_peaks, peak_widths
import matplotlib.pyplot as plt

In [2]:
# load data set
folder_path = Path(r"C:\Users\YourName\Desktop\your_tiff_folder")

# Optional: if you already have an explicit list of TIFF paths, put them here.
# Leave this empty to load every .tif/.tiff file from folder_path.
folder_path_list = []

if folder_path_list:
    tif_files = [Path(p) for p in folder_path_list]
else:
    tif_files = sorted(list(folder_path.glob("*.tif")) + list(folder_path.glob("*.tiff")))

print(f"Found {len(tif_files)} TIFF files")


Found 0 TIFF files


In [3]:
# remove cosmic rays for ONE spectrum
def clean_spectrum(spectrum, max_width=2):
    y = spectrum.copy()
    cosmic_mask = np.zeros_like(y, dtype=bool)

    while True:
        peaks, _ = find_peaks(y)

        if len(peaks) == 0:
            break

        peak_heights = y[peaks]
        highest_peak_idx = peaks[np.argmax(peak_heights)]

        widths = peak_widths(y, [highest_peak_idx], rel_height=0.5)[0]
        width = widths[0]

        if width > max_width:
            break

        left = highest_peak_idx - 1
        right = highest_peak_idx + 1

        cosmic_mask[highest_peak_idx] = True

        if left >= 0 and right < len(y):
            y[highest_peak_idx] = (y[left] + y[right]) / 2
        elif left >= 0:
            y[highest_peak_idx] = y[left]
        elif right < len(y):
            y[highest_peak_idx] = y[right]
        else:
            break

    return y, cosmic_mask

In [4]:
# loop through every TIFF file
all_pixel_maxima = []          
per_file_max_maps = {}         

for tif_file in tif_files:
    print(f"Processing: {tif_file.name}")

    img = tiff.imread(tif_file)
    print("  shape:", img.shape)

    # -----------------------------------
    # IMPORTANT:
    # assume TIFF is already shaped like:
    # (nx, nspec, ny)
    #
    # if not, reorder it here
    # example:
    # img = np.moveaxis(img, 2, 1)
    # if original shape is (nx, ny, nspec)
    # -----------------------------------
    spectra_data = img

    if spectra_data.ndim != 3:
        print(f"  Skipping {tif_file.name}: not a 3D Raman cube")
        continue

    nx, nspec, ny = spectra_data.shape

    cleaned_cube = np.zeros_like(spectra_data)
    cosmic_mask_cube = np.zeros_like(spectra_data, dtype=bool)

    for i in range(nx):
        for j in range(ny):
            spectrum = spectra_data[i, :, j]
            cleaned_spectrum, cosmic_mask = clean_spectrum(spectrum, max_width=2)
            cleaned_cube[i, :, j] = cleaned_spectrum
            cosmic_mask_cube[i, :, j] = cosmic_mask

    max_intensity_clean = np.max(cleaned_cube, axis=1) # need to ask michael about this

    per_file_max_maps[tif_file.name] = max_intensity_clean

    all_pixel_maxima.append(max_intensity_clean.flatten())

In [5]:
# Graphing 1. combine all pixel maxima into one array
all_pixel_maxima = np.concatenate(all_pixel_maxima)

print("\nTotal number of pixel maxima collected:", len(all_pixel_maxima))
print("Overall maximum across all files:", np.max(all_pixel_maxima))

plt.figure(figsize=(8, 5))
plt.hist(all_pixel_maxima, bins=50)
plt.xlabel("Maximum Intensity")
plt.ylabel("Number of Pixels")
plt.title("Distribution of Maximum Intensity per Pixel Across All TIFF Files")
plt.tight_layout()
plt.show()


ValueError: need at least one array to concatenate

In [ ]:
# Task cell completed:
# 1) clean cosmic rays
# 2) isolate wavelengths 1600-3000 cm^-1
# 3) save filtered TIFF cubes to a new folder
# 4) show how to plot (a) one pixel spectrum and (b) one wavelength image

# -----------------------------
# REQUIRED INPUT
# -----------------------------
# You need a 1D wavenumber axis with length = nspec.
# Replace this example with your real Raman shift values.
# Example if you have them in a CSV or TXT file:
# wavenumbers = np.loadtxt(r"C:\Users\YourName\Desktop\wavenumbers.txt")
# OR:
# wavenumbers = np.loadtxt(r"C:\Users\YourName\Desktop\wavenumbers.csv", delimiter=",")

# Placeholder example (EDIT THIS):
# wavenumbers = np.linspace(400, 3200, nspec)

output_folder = folder_path / "cleaned_1600_3000"
output_folder.mkdir(exist_ok=True)

# dictionaries to store results if you want to inspect them later
filtered_cubes = {}
filtered_wavenumbers = None

for tif_file in tif_files:
    print(f"\nProcessing task cell for: {tif_file.name}")

    img = tiff.imread(tif_file)
    spectra_data = img

    if spectra_data.ndim != 3:
        print(f"  Skipping {tif_file.name}: not a 3D Raman cube")
        continue

    nx, nspec, ny = spectra_data.shape

    # ---------- make sure wavenumbers exists and matches nspec ----------
    try:
        wavenumbers
    except NameError:
        raise ValueError(
            "Define `wavenumbers` before running this cell. "
            "It must be a 1D array with length equal to nspec."
        )

    wavenumbers = np.asarray(wavenumbers)
    if wavenumbers.ndim != 1 or len(wavenumbers) != nspec:
        raise ValueError(
            f"wavenumbers must be 1D with length {nspec}, but got shape {wavenumbers.shape}"
        )

    # ---------- clean cosmic rays pixel by pixel ----------
    cleaned_cube = np.zeros_like(spectra_data)
    cosmic_mask_cube = np.zeros_like(spectra_data, dtype=bool)

    for i in range(nx):
        for j in range(ny):
            spectrum = spectra_data[i, :, j]
            cleaned_spectrum, cosmic_mask = clean_spectrum(spectrum, max_width=2)
            cleaned_cube[i, :, j] = cleaned_spectrum
            cosmic_mask_cube[i, :, j] = cosmic_mask

    # ---------- isolate spectral range 1600-3000 cm^-1 ----------
    keep_mask = (wavenumbers >= 1600) & (wavenumbers <= 3000)
    if not np.any(keep_mask):
        raise ValueError("No wavelengths found between 1600 and 3000 cm^-1")

    filtered_cube = cleaned_cube[:, keep_mask, :]
    filtered_wn = wavenumbers[keep_mask]

    # store in memory
    filtered_cubes[tif_file.name] = filtered_cube
    filtered_wavenumbers = filtered_wn

    # ---------- save filtered cube ----------
    out_path = output_folder / f"{tif_file.stem}_cleaned_1600_3000.tif"
    tiff.imwrite(out_path, filtered_cube.astype(cleaned_cube.dtype))
    print(f"  Saved: {out_path.name}")
    print(f"  Original shape: {spectra_data.shape}")
    print(f"  Filtered shape: {filtered_cube.shape}")

print("\nDone.")
print(f"Filtered TIFF files saved in: {output_folder}")

# =========================================================
# EXAMPLE A: plot the spectrum of ONE pixel from ONE TIFF file
# =========================================================
# Choose file index and pixel coordinates
example_file = tif_files[0]      # first TIFF file
pixel_x = 0                      # edit this
pixel_y = 0                      # edit this

# reload saved filtered TIFF for plotting
example_cube = tiff.imread(output_folder / f"{example_file.stem}_cleaned_1600_3000.tif")
pixel_spectrum = example_cube[pixel_x, :, pixel_y]

plt.figure(figsize=(8, 5))
plt.plot(filtered_wavenumbers, pixel_spectrum)
plt.xlabel("Raman Shift (cm$^{-1}$)")
plt.ylabel("Intensity")
plt.title(f"Pixel spectrum: {example_file.name} at (x={pixel_x}, y={pixel_y})")
plt.tight_layout()
plt.show()

# =========================================================
# EXAMPLE B: draw ONE wavelength image from ONE TIFF file
# =========================================================
# Pick the wavelength you want to display
wanted_wavenumber = 1650   # edit this

# find nearest available spectral index after filtering
k = np.argmin(np.abs(filtered_wavenumbers - wanted_wavenumber))
closest_wavenumber = filtered_wavenumbers[k]
wavelength_image = example_cube[:, k, :]

plt.figure(figsize=(6, 6))
plt.imshow(wavelength_image, cmap="inferno")
plt.colorbar(label="Intensity")
plt.title(f"{example_file.name}\nImage at {closest_wavenumber:.2f} cm$^{{-1}}$")
plt.xlabel("y pixel")
plt.ylabel("x pixel")
plt.tight_layout()
plt.show()

# =========================================================
# OPTIONAL: if you want one pixel from the ORIGINAL full spectrum
# =========================================================
original_cube = tiff.imread(example_file)
original_pixel_spectrum = original_cube[pixel_x, :, pixel_y]

plt.figure(figsize=(8, 5))
plt.plot(wavenumbers, original_pixel_spectrum)
plt.xlabel("Raman Shift (cm$^{-1}$)")
plt.ylabel("Intensity")
plt.title(f"Original full spectrum: {example_file.name} at (x={pixel_x}, y={pixel_y})")
plt.tight_layout()
plt.show()
